In [6]:
import time
import pandas as pd
import requests

headers = {
    "User-Agent": "WikidataResearch/1.0 (academic research; 1823315676@qq.com)"
}

df = pd.read_csv("sample1_15000.csv")
df["QID"] = df["item"].str.extract(r"(Q\d+)")

#df = df.drop_duplicates(subset=["QID"]).reset_index(drop=True)
#qids = df["QID"].dropna().tolist()
#print(f"Loaded {len(qids)} entities.")


# Fetch Wikidata entities by ID
def fetch_entities(ids, props, max_retries=3):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(ids),
        "format": "json",
        "languages": "en",
        "props": props
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=60)
            return response.json()
        except Exception as e:
            print(f"Request failed, attempt {attempt + 1}/{max_retries}: {e}")
            time.sleep(2 ** attempt)

    return None


# Fetch entity labels and claims
all_entities = {}
batch_size = 50
qid_batches = [qids[i:i + batch_size] for i in range(0, len(qids), batch_size)]

print(f"Fetching entity data in {len(qid_batches)} batches...")

for i, batch in enumerate(qid_batches, start=1):
    if i == 1 or i % 20 == 0 or i == len(qid_batches):
        print(f"Processing entity batch {i}/{len(qid_batches)}")

    data = fetch_entities(batch, props="labels|claims")

    if data and "entities" in data:
        all_entities.update(data["entities"])

    time.sleep(0.3)

print(f"Finished. Fetched data for {len(all_entities)} entities.")


# Collect all property IDs and fetch property labels
all_property_ids = set()

for qid, entity in all_entities.items():
    claims = entity.get("claims", {})

    for property_id, statements in claims.items():
        # Main property
        all_property_ids.add(property_id)

        for statement in statements:
            # Qualifier properties
            for qualifier_property_id in statement.get("qualifiers", {}).keys():
                all_property_ids.add(qualifier_property_id)

            # Reference properties
            for reference in statement.get("references", []):
                for reference_property_id in reference.get("snaks", {}).keys():
                    all_property_ids.add(reference_property_id)

print(f"Found {len(all_property_ids)} distinct properties. Fetching property labels...")

property_labels = {}
property_list = sorted(all_property_ids)
property_batches = [property_list[i:i + batch_size] for i in range(0, len(property_list), batch_size)]

for i, batch in enumerate(property_batches, start=1):
    if i == 1 or i % 20 == 0 or i == len(property_batches):
        print(f"Processing property batch {i}/{len(property_batches)}")

    data = fetch_entities(batch, props="labels")

    if data and "entities" in data:
        for pid, entity in data["entities"].items():
            label = entity.get("labels", {}).get("en", {}).get("value", pid)
            property_labels[pid] = label

    time.sleep(0.2)

print(f"Finished. Fetched labels for {len(property_labels)} properties.")


# Return the QID/PID used as a snak value, if the snak has a Wikibase entity value
def get_wikibase_id_from_snak(snak):
    datavalue = snak.get("datavalue", {})

    if datavalue.get("type") == "wikibase-entityid":
        return datavalue.get("value", {}).get("id", "")

    return ""


value_entity_ids = set()

for qid, entity in all_entities.items():
    claims = entity.get("claims", {})

    for property_id, statements in claims.items():
        for statement in statements:
            # Main value
            value_id = get_wikibase_id_from_snak(statement.get("mainsnak", {}))
            if value_id:
                value_entity_ids.add(value_id)

            # Qualifier values
            for qualifier_snaks in statement.get("qualifiers", {}).values():
                for qualifier_snak in qualifier_snaks:
                    value_id = get_wikibase_id_from_snak(qualifier_snak)
                    if value_id:
                        value_entity_ids.add(value_id)

            # Reference values
            for reference in statement.get("references", []):
                for reference_snaks in reference.get("snaks", {}).values():
                    for reference_snak in reference_snaks:
                        value_id = get_wikibase_id_from_snak(reference_snak)
                        if value_id:
                            value_entity_ids.add(value_id)

print(f"Found {len(value_entity_ids)} Wikibase entity values. Fetching value labels...")

value_labels = {}
value_entity_list = sorted(value_entity_ids)
value_batches = [value_entity_list[i:i + batch_size] for i in range(0, len(value_entity_list), batch_size)]

for i, batch in enumerate(value_batches, start=1):
    if i % 20 == 0:
        print(f"Processing value label batch {i}/{len(value_batches)}")

    data = fetch_entities(batch, props="labels")

    if data and "entities" in data:
        for value_id, entity in data["entities"].items():
            label = entity.get("labels", {}).get("en", {}).get("value", value_id)
            value_labels[value_id] = label

    time.sleep(0.2)

print(f"Finished. Fetched labels for {len(value_labels)} Wikibase entity values.")



# Find the maximum number of qualifiers in one statement
max_qualifiers = 0

for qid, entity in all_entities.items():
    for property_id, statements in entity.get("claims", {}).items():
        for statement in statements:
            qualifier_count = sum(len(snaks) for snaks in statement.get("qualifiers", {}).values())
            max_qualifiers = max(max_qualifiers, qualifier_count)

print(f"Maximum number of qualifiers in one statement: {max_qualifiers}")


# Extract value ID and value label from a snak
def get_snak_value(snak):
    datavalue = snak.get("datavalue", {})
    value_type = datavalue.get("type", "")
    value = datavalue.get("value", "")

    if value_type == "wikibase-entityid":
        value_id = value.get("id", "")
        value_label = value_labels.get(value_id, property_labels.get(value_id, value_id))
        return value_id, value_label

    if value_type == "string":
        return value, value

    if value_type == "monolingualtext":
        text = value.get("text", "")
        return text, text

    if value_type == "quantity":
        amount = value.get("amount", "")
        return amount, amount

    if value_type == "time":
        time_value = value.get("time", "")
        return time_value, time_value

    if value_type == "globecoordinate":
        coordinate = f"{value.get('latitude', '')},{value.get('longitude', '')}"
        return coordinate, coordinate

    return str(value), str(value)


# Extract statement level data
rows = []

for qid, entity in all_entities.items():
    entity_label = entity.get("labels", {}).get("en", {}).get("value", qid)
    claims = entity.get("claims", {})

    for property_id, statements in claims.items():
        property_label = property_labels.get(property_id, property_id)
        total_statements = len(statements)

        for statement_index, statement in enumerate(statements, start=1):
            row = {}

            # Entity and main statement information
            row["QID"] = qid
            row["Entity_Label"] = entity_label
            row["Property_ID"] = property_id
            row["Property_Label"] = property_label
            row["Total_Statements"] = total_statements
            row["Statement_Number"] = statement_index
            row["Rank"] = statement.get("rank", "")

            # Main value
            mainsnak = statement.get("mainsnak", {})
            main_value_id, main_value_label = get_snak_value(mainsnak)

            row["Value_Type"] = mainsnak.get("datatype", "")
            row["Main_Value_ID"] = main_value_id
            row["Main_Value_Label"] = main_value_label

            # Qualifiers
            qualifiers = statement.get("qualifiers", {})
            qualifier_order = statement.get("qualifiers-order", list(qualifiers.keys()))

            qualifier_list = []

            for qualifier_property_id in qualifier_order:
                for qualifier_snak in qualifiers.get(qualifier_property_id, []):
                    qualifier_value_id, qualifier_value_label = get_snak_value(qualifier_snak)

                    qualifier_list.append({
                        "property_id": qualifier_property_id,
                        "property_label": property_labels.get(qualifier_property_id, qualifier_property_id),
                        "value_id": qualifier_value_id,
                        "value_label": qualifier_value_label
                    })

            row["Has_Qualifiers"] = "Yes" if qualifier_list else "No"
            row["Qualifier_Count"] = len(qualifier_list)

            for i in range(max_qualifiers):
                if i < len(qualifier_list):
                    row[f"Qualifier_Property_ID_{i + 1}"] = qualifier_list[i]["property_id"]
                    row[f"Qualifier_Property_Label_{i + 1}"] = qualifier_list[i]["property_label"]
                    row[f"Qualifier_Value_ID_{i + 1}"] = qualifier_list[i]["value_id"]
                    row[f"Qualifier_Value_Label_{i + 1}"] = qualifier_list[i]["value_label"]
                else:
                    row[f"Qualifier_Property_ID_{i + 1}"] = ""
                    row[f"Qualifier_Property_Label_{i + 1}"] = ""
                    row[f"Qualifier_Value_ID_{i + 1}"] = ""
                    row[f"Qualifier_Value_Label_{i + 1}"] = ""

            # References
            references = statement.get("references", [])

            row["Has_References"] = "Yes" if references else "No"
            row["Reference_Count"] = len(references)

            reference_parts = []

            for reference in references:
                for reference_property_id, reference_snaks in reference.get("snaks", {}).items():
                    reference_property_label = property_labels.get(reference_property_id, reference_property_id)

                    for reference_snak in reference_snaks:
                        reference_value_id, reference_value_label = get_snak_value(reference_snak)

                        reference_parts.append(
                            f"{reference_property_id}="
                            f"{reference_property_label}="
                            f"{reference_value_id}="
                            f"{reference_value_label}"
                        )

            row["Reference_Details"] = " | ".join(reference_parts)

            rows.append(row)



print(f"Extracted {len(rows)} statement rows. Saving the result...")

result_df = pd.DataFrame(rows)

output_file = "15000business_statements.xlsx"
result_df.to_excel(output_file, index=False)

print(f"Saved to {output_file}")
print(f"Number of rows: {len(result_df)}")
print(f"Number of entities: {result_df['QID'].nunique()}")
print(f"Number of properties: {result_df['Property_ID'].nunique()}")

Fetching entity data in 300 batches...
Processing entity batch 1/300
Processing entity batch 20/300
Processing entity batch 40/300
Processing entity batch 60/300
Processing entity batch 80/300
Processing entity batch 100/300
Processing entity batch 120/300
Processing entity batch 140/300
Processing entity batch 160/300
Processing entity batch 180/300
Processing entity batch 200/300
Processing entity batch 220/300
Processing entity batch 240/300
Processing entity batch 260/300
Processing entity batch 280/300
Processing entity batch 300/300
Finished. Retrieved data for 15000 entities.
Found 1776 distinct properties. Fetching property labels...
Processing property batch 1/36
Processing property batch 20/36
Processing property batch 36/36
Finished. Retrieved labels for 1776 properties.
Found 19262 Wikibase entity values. Fetching value labels...
Processing value label batch 20/386
Processing value label batch 40/386
Processing value label batch 60/386
Processing value label batch 80/386
Pr